# Phase 1: Snapshot Render API PNG (Rust Backend)

This notebook validates inline and file-path PNG delivery contracts on the Rust daemon, including output-path safety.

In [1]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('could not locate repository root from notebook cwd')


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests'))

import base64
import uuid
from io import BytesIO

import httpx
from PIL import Image

from lucida.client import LucidaClient
from conftest import create_render_omezarr
from rust_daemon import start_rust_daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-phase1-snapshot-rust-'))
dataset_uri = create_render_omezarr(str(tmp_dir / 'snapshot.zarr'))
daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=daemon.base_url, backend='rust')
print({'dataset_uri': dataset_uri, 'base_url': daemon.base_url})


{'dataset_uri': '/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-phase1-snapshot-rust-ea0l2uzs/snapshot.zarr', 'base_url': 'http://127.0.0.1:58793'}


In [3]:
opened = client.open_dataset(dataset_uri)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id)
view_id = created.view_state.view_id

inline = client.render_image(view_id=view_id, width_px=80, height_px=60)
assert inline.status == 'ok'
assert inline.images[0].mime == 'image/png'
img = Image.open(BytesIO(base64.b64decode(inline.images[0].bytes_base64))).convert('RGBA')
assert img.size == (80, 60)

{'view_id': view_id, 'state_version': inline.state_version, 'state_hash': inline.state_hash}


{'view_id': 'view_52e27b49d4344f34',
 'state_version': 0,
 'state_hash': '2a0c6c9ff90fdbfd86a1e21a87aac02c3d56f2e23e14e86b50b97b86fcda8879'}

In [4]:
output_relative = f'snapshots/phase1-snapshot-{uuid.uuid4().hex}.png'
file_render = client.render_image(
    view_id=view_id,
    width_px=64,
    height_px=48,
    delivery='file_path',
    file_path=output_relative,
)

file_path = Path(file_render.images[0].file_path)
output_root = REPO_ROOT / 'output'
assert file_path.exists()
assert file_path.is_relative_to(output_root)
assert file_path.suffix == '.png'

with httpx.Client(base_url=daemon.base_url, timeout=30.0) as http_client:
    invalid = http_client.post(
        '/render/image',
        json={
            'schema_version': 1,
            'view_id': view_id,
            'output': {
                'width_px': 64,
                'height_px': 48,
                'delivery': 'file_path',
                'file_path': '../escape.png',
            },
        },
    )

assert invalid.status_code == 422
invalid_payload = invalid.json()
assert invalid_payload['code'] == 'render_output_path_invalid'

{'file_path': str(file_path), 'invalid_code': invalid_payload['code']}


{'file_path': '/Users/austin/GitHub/lucida/output/snapshots/phase1-snapshot-ddec3756112249c795172736c18281b9.png',
 'invalid_code': 'render_output_path_invalid'}

In [5]:
if 'file_path' in globals() and file_path.exists():
    file_path.unlink()
if 'client' in globals():
    client.close()
if 'daemon' in globals():
    daemon.stop()
if 'tmp_dir' in globals():
    shutil.rmtree(tmp_dir, ignore_errors=True)
